# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score, classification_report

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
spaceship = spaceship.dropna() # Dropping nnull values
spaceship["Cabin"] = spaceship["Cabin"].str[0] # transforming variable Cabin
spaceship = spaceship.drop(columns=["PassengerId", "Name"]) # dropping variables

In [4]:
## New way for encoding categorical variables ##
num_cols = spaceship.select_dtypes(exclude="object").drop(columns=["Transported"]).columns
cat_cols = spaceship.select_dtypes(include="object").columns
encoder = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
encoded_cat = encoder.fit_transform(spaceship[cat_cols])

In [5]:
encoded_cat_df = pd.DataFrame(encoded_cat, 
                              columns=encoder.get_feature_names_out(cat_cols),
                              index=spaceship.index)
X_num = spaceship[num_cols]
X = pd.concat([X_num, encoded_cat_df], axis=1)

In [6]:
spaceship = pd.get_dummies(spaceship, drop_first=True) # Convert categorical variables into dummy variables. We obtain Boolean variables
# Define features and target
X2 = spaceship.drop(columns=["Transported"]) # Solution with Booleans
y = spaceship["Transported"]

In [7]:
# Training the model and normalizing (I may have forgotten this part in the previous lab, therefore, it is implemented now!)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
normalizer = MinMaxScaler()
X_train_norm = normalizer.fit_transform(X_train) # Shorter version
X_test_norm = normalizer.transform(X_test)

In [9]:
## Running the best model Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train_norm, y_train)
score = gb.score(X_test_norm, y_test)
print("Gradient Boosting accuracy with normalization =", score)

Gradient Boosting accuracy with normalization = 0.8071104387291982


In [10]:
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
gb_score = gb.score(X_test, y_test)
print("Gradient Boosting accuracy without normaliozation =", gb_score)

Gradient Boosting accuracy without normaliozation = 0.8071104387291982


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [11]:
# As we have a classification problem, we should impmente in another way the grid
grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4],                 #"max_leaf_nodes": [4, 8, 16, 32],  # Often set in powers of 2  
    "subsample": [0.8, 1.0]}


Most data scientists tune either max_depth or max_leaf_nodes, but rarely both at the same time, 
to avoid redundant training combinations.

- Run Grid Search

In [12]:
gb_grid = GradientBoostingClassifier(random_state=42)
grid_search = GridSearchCV(estimator=gb_grid, param_grid=grid, cv=5, n_jobs=-1)

In [13]:
grid_search.fit(X_train, y_train)

,estimator,GradientBoost...ndom_state=42)
,param_grid,"{'learning_rate': [0.01, 0.05, ...], 'max_depth': [2, 3, ...], 'n_estimators': [50, 100, ...], 'subsample': [0.8, 1.0]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


- Evaluate your model

In [14]:
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Best CV score: 0.8012881296407788


In [15]:
best_gb = grid_search.best_estimator_
best_score = best_gb.score(X_test, y_test)
print("Best Gradient Boosting test accuracy =", best_score)

Best Gradient Boosting test accuracy = 0.8139183055975794


Grid Search improved the Gradient Boosting model by testing different combinations of hyperparameters. The best model achieved an accuracy of approximately 81.4%, which is higher than the previous models. The selected parameters helped balance model complexity and generalization, improving predictive performance on unseen data.

In [16]:
train_score = best_gb.score(X_train, y_train)
print("Training accuracy =", train_score)
test_score = best_gb.score(X_test, y_test)
print("Test accuracy =", test_score)


Training accuracy = 0.8177517032551098
Test accuracy = 0.8139183055975794


The model does not show signs of overfitting because the training accuracy (0.8178) and test accuracy (0.8139) are very close. This indicates good generalization performance on unseen data.